In [1]:
import flickrapi
import pandas as pd
import folium
import numpy as np
import os
from dotenv import load_dotenv 

# =========================================================
# 💡 APIキーの安全な読み込み
load_dotenv() 

api_key = os.getenv('FLICKR_API_KEY')
api_secret = os.getenv('FLICKR_API_SECRET')

if not api_key or not api_secret:
    print("⚠️ API認証情報が見つかりません。.envファイルを確認してください。")
else:
    print("✅ API認証情報を読み込みました。")
# =========================================================

# connect to flickr
flickr = flickrapi.FlickrAPI(api_key, api_secret, format="parsed-json")

# --- 新しい指定範囲の座標設定 ---
lat_north = 35.65435
lat_south = 35.64349
long_east = 139.70442
long_west = 139.68912

# 中心座標の計算
centre_latitude = (lat_north + lat_south) / 2
centre_longitude = (long_east + long_west) / 2

per_page = 500

# --- バウンディングボックスの分割関数 ---
def split_bbox(north, south, west, east, parts=3):
    lat_steps = np.linspace(south, north, parts + 1)
    lon_steps = np.linspace(west, east, parts + 1)
    bbox_list = []
    for i in range(parts):
        for j in range(parts):
            min_lat, max_lat = lat_steps[i], lat_steps[i+1]
            min_lon, max_lon = lon_steps[j], lon_steps[j+1]
            bbox_list.append(f"{min_lon},{min_lat},{max_lon},{max_lat}")
    return bbox_list

# 3x3の9エリアに分割
SUB_BBOXES = split_bbox(lat_north, lat_south, long_west, long_east, parts=3)
print(f"✅ 全エリアを {len(SUB_BBOXES)} 個のサブボックスに分割しました。")

✅ API認証情報を読み込みました。
✅ 全エリアを 9 個のサブボックスに分割しました。


In [2]:
import numpy as np
import re 
from tqdm import tqdm 

CSV_FILENAME = "Tokyo_Flickr_Clean.csv"
HTML_FILENAME = "Tokyo_Flickr_Map_Total.html"
MISSING_LABEL = "N/A"
all_data_frames = []
MAX_PAGES_PER_SUB_BOX = 10 # 1サブボックスあたり最大5000枚取得

def get_page(bbox_str, page, per_page):
    response = flickr.photos.search(
        bbox=bbox_str,
        per_page=per_page,
        page=page,
        has_geo=1,
        extras="geo,description,tags,url_s,owner_name,count_comments",
    )
    photos = response["photos"]["photo"]
    rows = []
    for photo in photos:
        desc_content = photo.get("description", {}).get("_content", "")
        lat, lon = photo.get("latitude"), photo.get("longitude")
        rows.append({
            "id": photo.get("id", MISSING_LABEL),
            "title": photo.get("title", ""),
            "description": desc_content,
            "latitude": float(lat) if lat else np.nan,
            "longitude": float(lon) if lon else np.nan,
            "url_s": photo.get("url_s", MISSING_LABEL),
            "owner_name": photo.get("ownername", MISSING_LABEL),
            "tags": photo.get("tags", ""),
        })
    return pd.DataFrame(rows)

def clean_html_tags(text):
    if pd.isna(text): return text
    return re.sub('<.*?>', '', str(text)).strip()

# --- メイン実行: 分割取得 ---
if 'SUB_BBOXES' in locals():
    for idx, bbox_str in enumerate(tqdm(SUB_BBOXES, desc="Sub-Boxes取得中")):
        try:
            res = flickr.photos.search(bbox=bbox_str, per_page=per_page, page=1, has_geo=1)
            pages = min(res["photos"]["pages"], MAX_PAGES_PER_SUB_BOX)
            for page in range(1, pages + 1):
                all_data_frames.append(get_page(bbox_str, page, per_page))
        except Exception as e:
            print(f"Error at Sub-Box {idx}: {e}")

    if all_data_frames:
        df = pd.concat(all_data_frames, ignore_index=True).drop_duplicates(subset=['id'])
        df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
        df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
        df['description'] = df['description'].apply(clean_html_tags)
        
        map_df = df.dropna(subset=['latitude', 'longitude']).copy()
        map_df.to_csv(CSV_FILENAME, index=False)

        # --- マッピング ---
        m = folium.Map(location=[centre_latitude, centre_longitude], tiles="CartoDB positron", zoom_start=16)
        folium.Rectangle(bounds=[[lat_north, long_west], [lat_south, long_east]], color="red", weight=2, fill=False).add_to(m)

        for _, row in map_df.iterrows():
            folium.CircleMarker(
                location=[row["latitude"], row["longitude"]],
                radius=2, color="#0072b2", fill=True, popup=row["title"]
            ).add_to(m)
        
        m.save(HTML_FILENAME)
        print(f"✅ 完了！計 {len(map_df)} 枚の写真を '{HTML_FILENAME}' に出力しました。")

Sub-Boxes取得中: 100%|██████████| 9/9 [05:27<00:00, 36.42s/it]


✅ 完了！計 12608 枚の写真を 'Tokyo_Flickr_Map_Total.html' に出力しました。


In [1]:
import pandas as pd
from textblob import TextBlob
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import folium
import torch
import requests
from PIL import Image
from io import BytesIO
import sys
import re
from transformers import CLIPProcessor, CLIPModel
from tqdm import tqdm

# ---------------------------------------------------------
# 0. 設定：ファイル名と座標
# ---------------------------------------------------------
INPUT_FILE = 'Tokyo_Flickr_Clean.csv'
OUTPUT_FILE = 'Tokyo_Sentiment_CLIP_Final.csv'
MODEL_ID = "openai/clip-vit-base-patch32"

# 先ほど設定した範囲の中心座標
centre_latitude = 35.64892
centre_longitude = 139.69677

# VADERの準備
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon')

# ---------------------------------------------------------
# 1. 画像感情分析 (CLIP) のための東京用プロンプト設定
# ---------------------------------------------------------
POSITIVE_PROMPTS = [
    "vibrant and exciting Shibuya crossing",
    "peaceful and stylish Daikanyama cafe",
    "modern and beautiful Tokyo architecture",
    "clean and organized city street",
    "happy people enjoying Tokyo city life",
    "fashionable people in Daikanyama",
    "beautiful night view of Tokyo lights"
]

NEGATIVE_PROMPTS = [
    "overcrowded and stressful street",
    "dirty alleyway with trash",
    "gloomy and grey rainy city",
    "noisy construction site with scaffolding",
    "messy and unorganized urban area",
    "blurred low quality photo",
    "lonely and depressed city atmosphere"
]

ALL_PROMPTS = POSITIVE_PROMPTS + NEGATIVE_PROMPTS

# CLIPモデルのロード
print(f"Loading CLIP model ({MODEL_ID})...")
model = CLIPModel.from_pretrained(MODEL_ID)
processor = CLIPProcessor.from_pretrained(MODEL_ID)

# ---------------------------------------------------------
# 2. 分析用ヘルパー関数
# ---------------------------------------------------------

def get_combined_text(row):
    # 分析対象のテキストを結合
    parts = [str(row.get(c, "")) for c in ['title', 'description', 'tags']]
    text = " ".join(parts).replace("N/A", "").strip()
    return text

sid = SentimentIntensityAnalyzer()

def analyze_image_clip(url):
    if not url or url == 'N/A' or not str(url).startswith('http'):
        return "N/A", 0.0
    try:
        response = requests.get(url, stream=True, timeout=5)
        image = Image.open(BytesIO(response.content)).convert("RGB")
        inputs = processor(text=ALL_PROMPTS, images=image, return_tensors="pt", padding=True)
        with torch.no_grad():
            outputs = model(**inputs)
        probs = outputs.logits_per_image.softmax(dim=1)[0].tolist()
        
        pos_score = sum(probs[:len(POSITIVE_PROMPTS)])
        neg_score = sum(probs[len(POSITIVE_PROMPTS):])
        
        best_desc = ALL_PROMPTS[probs.index(max(probs))]
        return best_desc, round(pos_score - neg_score, 4)
    except:
        return "Error", 0.0

# ---------------------------------------------------------
# 3. 分析実行
# ---------------------------------------------------------
print(f"\nReading {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)

print("Applying Text Analysis (TextBlob & VADER)...")
df['Combined_Text'] = df.apply(get_combined_text, axis=1)
df['TextBlob_Score'] = df['Combined_Text'].apply(lambda x: TextBlob(x).sentiment.polarity if x else 0.0)
df['Vader_Score'] = df['Combined_Text'].apply(lambda x: sid.polarity_scores(x)['compound'] if x else 0.0)

print(f"Applying Visual Analysis (CLIP) for {len(df)} images...")
clip_results = [analyze_image_clip(url) for url in tqdm(df['url_s'])]
df['CLIP_Best_Desc'], df['CLIP_Visual_Score'] = zip(*clip_results)

# CSV保存
df.to_csv(OUTPUT_FILE, index=False)

# ---------------------------------------------------------
# 4. マッピング関数の定義と出力
# ---------------------------------------------------------
def sentiment_to_color(value):
    # スコア(-1〜1)を赤(負)〜黄(中)〜緑(正)に変換
    norm = (value + 1) / 2
    r = int(255 * (1 - norm))
    g = int(255 * norm)
    return f"#{r:02x}{g:02x}00"

def create_map(df, score_col, filename):
    m = folium.Map(location=[centre_latitude, centre_longitude], tiles="Cartodb dark_matter", zoom_start=16)
    for _, row in df.iterrows():
        color = sentiment_to_color(row[score_col])
        popup_text = f"Title: {row['title']}<br>Score: {row[score_col]:.3f}<br><img src='{row['url_s']}' width='200'>"
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=4, color=color, fill=True, fill_opacity=0.8,
            popup=folium.Popup(popup_text, max_width=250)
        ).add_to(m)
    m.save(filename)
    print(f"Saved: {filename}")

print("\nGenerating Sentiment Maps...")
create_map(df, 'TextBlob_Score', 'map_Tokyo_Sentiment_Text.html')
create_map(df, 'CLIP_Visual_Score', 'map_Tokyo_Sentiment_Visual.html')

print("\n🎉 全ての感情分析とマッピングが完了しました！")

Loading CLIP model (openai/clip-vit-base-patch32)...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Exception ignored while calling deallocator <function tqdm.__del__ at 0x00000216801C0BF0>:
Traceback (most recent call last):
  File "c:\Users\taiga\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "c:\Users\taiga\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeErr


Reading Tokyo_Flickr_Clean.csv...
Applying Text Analysis (TextBlob & VADER)...
Applying Visual Analysis (CLIP) for 12608 images...


100%|██████████| 12608/12608 [1:53:50<00:00,  1.85it/s]     



Generating Sentiment Maps...
Saved: map_Tokyo_Sentiment_Text.html
Saved: map_Tokyo_Sentiment_Visual.html

🎉 全ての感情分析とマッピングが完了しました！


In [2]:
import pandas as pd
import numpy as np

def fix_csv_alignment(input_file, output_file):
    # いったん全データを読み込む
    df = pd.read_csv(input_file)
    fixed_rows = []

    for index, row in df.iterrows():
        # 緯度(latitude)と経度(longitude)の候補を抽出
        # 本来の列だけでなく、ズレが生じやすい隣の列もチェックします
        vals = row.values.tolist()
        
        lat = np.nan
        lon = np.nan
        
        # リスト全体から「緯度(35.x)」と「経度(139.x)」っぽい数値を探すロジック
        for v in vals:
            try:
                f_val = float(v)
                if 35.0 <= f_val <= 36.0: # 東京の緯度範囲
                    lat = f_val
                elif 139.0 <= f_val <= 140.0: # 東京の経度範囲
                    lon = f_val
            except (ValueError, TypeError):
                continue
        
        # 修正された値を反映
        row['latitude'] = lat
        row['longitude'] = lon
        
        # URLが緯度列に入っている場合の対策（url_s列がN/Aで、lat列にhttpがある場合など）
        if 'http' in str(row['latitude']):
             row['url_s'] = row['latitude']
             row['latitude'] = lat # 上で見つけた数値を入れる

        fixed_rows.append(row)

    # 新しいデータフレームを作成
    fixed_df = pd.DataFrame(fixed_rows)
    
    # 緯度・経度がどうしても見つからなかった行を削除（マッピング不能なため）
    clean_df = fixed_df.dropna(subset=['latitude', 'longitude'])
    
    # 保存
    clean_df.to_csv(output_file, index=False)
    print(f"✅ 修正完了！")
    print(f"元の行数: {len(df)} -> 修正後の有効行数: {len(clean_df)}")
    print(f"保存先: {output_file}")

# 実行
fix_csv_alignment('Tokyo_Sentiment_CLIP_Final.csv', 'Tokyo_Flickr_Fixed.csv')

✅ 修正完了！
元の行数: 12608 -> 修正後の有効行数: 12608
保存先: Tokyo_Flickr_Fixed.csv


In [3]:
import pandas as pd
import re

def robust_fix_csv(input_file, output_file):
    # すべての行をテキストとして読み込む（構造が壊れている可能性があるため）
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    header = lines[0].strip().split(',')
    fixed_data = []

    for line in lines[1:]:
        # カンマで分割するが、引用符内のカンマを保護
        # （簡易的な正規表現でパース）
        parts = re.findall(r'(?:[^,"]|"(?:\\.|[^"])*")+', line.strip())
        
        row_dict = {h: "N/A" for h in header}
        
        lat, lon, score, url = None, None, None, None
        
        for p in parts:
            p = p.strip('"')
            # 緯度のパターン (35.xxx)
            if re.match(r'^35\.\d+$', p):
                lat = p
            # 経度のパターン (139.xxx)
            elif re.match(r'^139\.\d+$', p):
                lon = p
            # CLIPスコアのパターン (-0.xxxx)
            elif re.match(r'^-?\d\.\d{3,}$', p):
                score = p
            # URLのパターン
            elif p.startswith('http'):
                url = p
        
        # 見つかった値を強制的に正しい列へ
        if lat and lon:
            row_dict['latitude'] = lat
            row_dict['longitude'] = lon
            row_dict['CLIP_Visual_Score'] = score if score else 0.0
            row_dict['url_s'] = url if url else "N/A"
            fixed_data.append(row_dict)

    new_df = pd.DataFrame(fixed_data)
    new_df.to_csv(output_file, index=False)
    print(f"✅ 強化修正完了: {len(new_df)} 行を保存しました。")

robust_fix_csv('Tokyo_Sentiment_CLIP_Final.csv', 'Tokyo_Flickr_Fixed_v2.csv')

✅ 強化修正完了: 11757 行を保存しました。


In [1]:
import pandas as pd
import re

def save_numeric_sentiment_csv(input_file, output_file):
    # すべての行をテキストとして読み込む
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    header = lines[0].strip().split(',')
    fixed_numeric_data = []

    for line in lines[1:]:
        # 正規表現でカンマ分割（引用符内を保護）
        parts = re.findall(r'(?:[^,"]|"(?:\\.|[^"])*")+', line.strip())
        
        lat, lon, clip_score, tb_score, v_score = None, None, None, None, None
        
        for p in parts:
            p = p.strip('"').strip()
            # 緯度の抽出 (35.xxx)
            if re.match(r'^35\.\d+$', p):
                lat = float(p)
            # 経度の抽出 (139.xxx)
            elif re.match(r'^139\.\d+$', p):
                lon = float(p)
            # CLIPスコアの抽出 (浮動小数点数)
            elif re.match(r'^-?\d\.\d{3,}$', p) and clip_score is None:
                clip_score = float(p)
            # TextBlobやVaderスコアの抽出 (残りの数値)
            elif re.match(r'^-?\d\.\d+$', p):
                val = float(p)
                if tb_score is None: tb_score = val
                elif v_score is None: v_score = val
        
        # 緯度・経度が揃っている場合のみ保存（Grasshopperでのエラー防止）
        if lat and lon:
            fixed_numeric_data.append({
                'latitude': lat,
                'longitude': lon,
                'CLIP_Visual_Score': clip_score if clip_score is not None else 0.0,
                'TextBlob_Score': tb_score if tb_score is not None else 0.0,
                'Vader_Score': v_score if v_score is not None else 0.0
            })

    # 数値のみのデータフレームを作成
    df_numeric = pd.DataFrame(fixed_numeric_data)
    
    # 重複を削除
    df_numeric = df_numeric.drop_duplicates()
    
    # 保存
    df_numeric.to_csv(output_file, index=False)
    print(f"✅ 数値専用CSVの保存が完了しました: {output_file}")
    print(f"有効なデータ点数: {len(df_numeric)}")
    print("列構成: latitude, longitude, CLIP_Visual_Score, TextBlob_Score, Vader_Score")

# 実行
save_numeric_sentiment_csv('Tokyo_Sentiment_CLIP_Final.csv', 'Tokyo_Flickr_Numeric_Only.csv')

✅ 数値専用CSVの保存が完了しました: Tokyo_Flickr_Numeric_Only.csv
有効なデータ点数: 8210
列構成: latitude, longitude, CLIP_Visual_Score, TextBlob_Score, Vader_Score
